# 11 — File-grouped DataLoader throughput benchmark

Measure end-to-end training throughput with 0, 2, 4, and 8
workers on the same balanced set of 512 real 2013 files.
Each dataset item opens one NetCDF file and returns all four
frames.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.7"
FILES_PER_CLASS = 256
FILE_BATCH_SIZE = 16
WORKER_COUNTS = (0, 2, 4, 8)
BENCHMARK_PASSES = 2
SEED = 20260913

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        f"tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "2013_baseline_v1"
    / "normalization.json"
)
RESULT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "io_benchmark_v1"
)
RESULT_PATH = (
    RESULT_DIRECTORY
    / "2013_dataloader_benchmark.json"
)

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_2013_loader_benchmark"
)

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    ARCHIVE_PATH,
    NORMALIZATION_PATH,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required path: {path}"
        )

if RESULT_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite {RESULT_PATH}"
    )

print("package:", PACKAGE_PATH)
print("result:", RESULT_PATH)

package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl
result: /content/drive/MyDrive/TorNet_Backup/experiments/io_benchmark_v1/2013_dataloader_benchmark.json


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl'], returncode=0)

In [4]:
import json
import random
import shutil
import tarfile
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    load_canonical_frame_index,
    read_netcdf_file,
)

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select a Colab GPU runtime "
        "and run all cells"
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)
means = np.asarray(
    normalization["means"],
    dtype=np.float32,
)
stds = np.asarray(
    normalization["standard_deviations"],
    dtype=np.float32,
)

assert means.shape == (4,)
assert stds.shape == (4,)

index = load_canonical_frame_index(
    MANIFESTS_ROOT
)
eligible = index.loc[
    index["year"].eq(2013)
    & index["split"].eq("train")
].copy()

files = (
    eligible.groupby(
        "archive_member",
        as_index=False,
    )
    .agg(
        positive_frames=(
            "frame_label",
            "sum",
        )
    )
)

positive_files = (
    files.loc[
        files["positive_frames"].gt(0),
        "archive_member",
    ]
    .sort_values()
    .head(FILES_PER_CLASS)
    .tolist()
)
negative_files = (
    files.loc[
        files["positive_frames"].eq(0),
        "archive_member",
    ]
    .sort_values()
    .head(FILES_PER_CLASS)
    .tolist()
)

if len(positive_files) != FILES_PER_CLASS:
    raise AssertionError(
        f"Expected {FILES_PER_CLASS} "
        f"positive files; found "
        f"{len(positive_files)}"
    )

if len(negative_files) != FILES_PER_CLASS:
    raise AssertionError(
        f"Expected {FILES_PER_CLASS} "
        f"negative files; found "
        f"{len(negative_files)}"
    )

member_names = (
    positive_files + negative_files
)

expected_labels = (
    eligible.loc[
        eligible["archive_member"].isin(
            member_names
        )
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
    .groupby("archive_member")[
        "frame_label"
    ]
    .apply(
        lambda values: (
            values.astype(
                np.uint8
            ).to_numpy()
        )
    )
    .to_dict()
)

print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print("torch:", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print(
    "benchmark files:",
    len(member_names),
)
print(
    "benchmark frames:",
    len(member_names) * 4,
)

tornado_detection: 0.1.7
torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB
benchmark files: 512
benchmark frames: 2048


In [5]:
if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter() - copy_started
)

required_members = set(member_names)
extracted_members = set()
extraction_started = time.perf_counter()

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in required_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                f"Could not extract "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(
            member.name
        )

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing = (
    required_members
    - extracted_members
)

if missing:
    raise RuntimeError(
        f"Missing files: "
        f"{sorted(missing)[:10]}"
    )

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)
print(
    "extracted files:",
    len(extracted_members),
)

copy seconds: 5.636
extraction seconds: 9.556
extracted files: 512


In [6]:
class FileDataset(Dataset):
    def __init__(
        self,
        members,
        root,
        channel_means,
        channel_stds,
    ):
        self.members = list(members)
        self.root = root
        self.means = (
            channel_means.reshape(
                1,
                1,
                1,
                4,
            )
        )
        self.stds = (
            channel_stds.reshape(
                1,
                1,
                1,
                4,
            )
        )

    def __len__(self):
        return len(self.members)

    def __getitem__(self, item):
        member = self.members[item]
        result = read_netcdf_file(
            self.root / member
        )

        np.testing.assert_array_equal(
            result.labels,
            expected_labels[member],
        )

        values = (
            result.values - self.means
        ) / self.stds

        values = np.nan_to_num(
            values,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(
            np.float32,
            copy=False,
        )

        inputs = (
            torch.from_numpy(values)
            .permute(0, 3, 1, 2)
            .contiguous()
        )
        labels = torch.from_numpy(
            result.labels.astype(
                np.float32,
                copy=False,
            )
        ).reshape(4, 1)

        return inputs, labels


class RadarBaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                4,
                16,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, inputs):
        return self.classifier(
            self.features(inputs)
        )


dataset = FileDataset(
    member_names,
    EXTRACTION_ROOT,
    means,
    stds,
)
device = torch.device("cuda")
results = []

for worker_count in WORKER_COUNTS:
    generator = (
        torch.Generator()
        .manual_seed(SEED)
    )

    loader = DataLoader(
        dataset,
        batch_size=FILE_BATCH_SIZE,
        shuffle=True,
        num_workers=worker_count,
        pin_memory=True,
        persistent_workers=(
            worker_count > 0
        ),
        generator=generator,
    )

    model = RadarBaselineCNN().to(
        device
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
    )
    criterion = (
        nn.BCEWithLogitsLoss()
    )
    pass_results = []

    for pass_number in range(
        1,
        BENCHMARK_PASSES + 1,
    ):
        processed_frames = 0
        loss_total = 0.0
        model.train()

        torch.cuda.synchronize()
        started = time.perf_counter()

        for inputs, labels in loader:
            (
                file_count,
                frame_count,
            ) = inputs.shape[:2]

            inputs = inputs.reshape(
                file_count * frame_count,
                4,
                120,
                240,
            ).to(
                device,
                non_blocking=True,
            )
            labels = labels.reshape(
                file_count * frame_count,
                1,
            ).to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )
            logits = model(inputs)
            loss = criterion(
                logits,
                labels,
            )
            loss.backward()
            optimizer.step()

            processed_frames += int(
                labels.shape[0]
            )
            loss_total += (
                float(
                    loss.detach().cpu()
                )
                * int(labels.shape[0])
            )

        torch.cuda.synchronize()

        elapsed = (
            time.perf_counter()
            - started
        )

        pass_result = {
            "pass": pass_number,
            "seconds": elapsed,
            "frames": processed_frames,
            "frames_per_second": (
                processed_frames
                / elapsed
            ),
            "mean_loss": (
                loss_total
                / processed_frames
            ),
        }
        pass_results.append(
            pass_result
        )

        print(
            f"workers={worker_count} "
            f"pass={pass_number} "
            f"seconds={elapsed:.3f} "
            f"fps="
            f"{processed_frames / elapsed:.2f}"
        )

    measured = pass_results[-1]

    results.append(
        {
            "worker_count": (
                worker_count
            ),
            "file_batch_size": (
                FILE_BATCH_SIZE
            ),
            "frame_batch_size": (
                FILE_BATCH_SIZE * 4
            ),
            "passes": pass_results,
            "measured_pass": (
                BENCHMARK_PASSES
            ),
            "measured_seconds": (
                measured["seconds"]
            ),
            "measured_frames_per_second": (
                measured[
                    "frames_per_second"
                ]
            ),
        }
    )

workers=0 pass=1 seconds=18.193 fps=112.57
workers=0 pass=2 seconds=17.888 fps=114.49
workers=2 pass=1 seconds=9.172 fps=223.28
workers=2 pass=2 seconds=9.000 fps=227.55
workers=4 pass=1 seconds=4.860 fps=421.36
workers=4 pass=2 seconds=4.692 fps=436.44
workers=8 pass=1 seconds=3.539 fps=578.67
workers=8 pass=2 seconds=3.222 fps=635.63


In [7]:
import datetime

best = max(
    results,
    key=lambda row: (
        row[
            "measured_frames_per_second"
        ]
    ),
)

canonical_train_frames = 547_672

projected_epoch_seconds = (
    canonical_train_frames
    / best["measured_frames_per_second"]
)

artifact = {
    "artifact_kind": (
        "file_grouped_dataloader_benchmark"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": PACKAGE_VERSION,
    "year": 2013,
    "benchmark_file_count": (
        len(member_names)
    ),
    "benchmark_frame_count": (
        len(member_names) * 4
    ),
    "positive_file_count": (
        len(positive_files)
    ),
    "negative_file_count": (
        len(negative_files)
    ),
    "benchmark_passes": (
        BENCHMARK_PASSES
    ),
    "results": results,
    "best_worker_count": (
        best["worker_count"]
    ),
    "best_frames_per_second": (
        best[
            "measured_frames_per_second"
        ]
    ),
    "canonical_train_frame_count": (
        canonical_train_frames
    ),
    "projected_training_epoch_seconds": (
        projected_epoch_seconds
    ),
    "projected_training_epoch_minutes": (
        projected_epoch_seconds / 60.0
    ),
    "copy_seconds": copy_seconds,
    "extraction_seconds": (
        extraction_seconds
    ),
}

RESULT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_PATH.write_text(
    json.dumps(
        artifact,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        artifact,
        indent=2,
        sort_keys=True,
    )
)
print("wrote:", RESULT_PATH)

{
  "artifact_kind": "file_grouped_dataloader_benchmark",
  "benchmark_file_count": 512,
  "benchmark_frame_count": 2048,
  "benchmark_passes": 2,
  "best_frames_per_second": 635.6279310045211,
  "best_worker_count": 8,
  "canonical_train_frame_count": 547672,
  "copy_seconds": 5.635912986999756,
  "created_at_utc": "2026-09-13T22:50:42.908518+00:00",
  "extraction_seconds": 9.55620574400018,
  "negative_file_count": 256,
  "package_version": "0.1.7",
  "positive_file_count": 256,
  "projected_training_epoch_minutes": 14.360392647065318,
  "projected_training_epoch_seconds": 861.6235588239191,
  "results": [
    {
      "file_batch_size": 16,
      "frame_batch_size": 64,
      "measured_frames_per_second": 114.48965737018372,
      "measured_pass": 2,
      "measured_seconds": 17.88807868799995,
      "passes": [
        {
          "frames": 2048,
          "frames_per_second": 112.5732752902139,
          "mean_loss": 0.4894328871741891,
          "pass": 1,
          "seconds": 18.

In [8]:
shutil.rmtree(EXTRACTION_ROOT)
LOCAL_ARCHIVE_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()
assert RESULT_PATH.is_file()

print(
    "Removed all Colab-local "
    "benchmark artifacts"
)
print("Preserved:", RESULT_PATH)

Removed all Colab-local benchmark artifacts
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/io_benchmark_v1/2013_dataloader_benchmark.json
